# 02 — Model Training
Train the `HopperMLP` dynamics model on real hardware data.

**Inputs:** `.mat` files in `DATA_DIR`  
**Outputs:** `experiments/weights/<run_name>.pt`, normalization stats

**Run after:** `01_data_exploration.ipynb`  
**Run before:** `03_simulation.ipynb`

## 1. Imports

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from hopper import (
    load_jumping_data,
    HopperDataset,
    HopperMLP,
    print_dataset_summary,
)
from hopper.evaluation import (
    compute_metrics,
    evaluate_rollout,
    plot_training_history,
    plot_rollout_predictions,
    print_model_summary,
)
from hopper.mpc import set_normalization_stats

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 2. Configuration

In [ ]:
# Paths
DATA_DIR    = "/Users/cassandrahe/MIT Dropbox/Cassandra He/jumping_data"
WEIGHTS_DIR = "../experiments/weights"
RUN_NAME    = "hopper_mlp_v1"  # Change this for each new experiment

os.makedirs(WEIGHTS_DIR, exist_ok=True)
MODEL_WEIGHTS = os.path.join(WEIGHTS_DIR, f"{RUN_NAME}.pt")
NORM_STATS    = os.path.join(WEIGHTS_DIR, f"{RUN_NAME}_norm.npz")

# Architecture
HIDDEN_DIM = 32

# Training
BATCH_SIZE    = 32
NUM_EPOCHS    = 100
LEARNING_RATE = 1e-3
LR_PATIENCE   = 10   # Epochs before LR reduction
LR_FACTOR     = 0.5

print(f"Run name:  {RUN_NAME}")
print(f"Weights -> {MODEL_WEIGHTS}")

## 3. Load & Split Data

**Important:** we use a temporal split (last 20% as test), not a random
shuffle, because this is time-series data. Random splitting leaks future
states into training.

In [ ]:
all_data = load_jumping_data(
    data_dir=DATA_DIR,
    downsample_factor=5,
    clip_start_sec=3.0,
    clip_end_sec=13.0,
)

X = all_data.X  # [N, 14]
Y = all_data.Y  # [N, 6]

# Temporal split: train on first 80%, validate on next 10%, test on last 10%
N = len(X)
train_end = int(0.8 * N)
val_end   = int(0.9 * N)

X_train, Y_train = X[:train_end],       Y[:train_end]
X_val,   Y_val   = X[train_end:val_end], Y[train_end:val_end]
X_test,  Y_test  = X[val_end:],          Y[val_end:]

print(f"Train: {len(X_train):,} samples")
print(f"Val:   {len(X_val):,} samples")
print(f"Test:  {len(X_test):,} samples")

## 4. Normalization
Compute stats from training data only, then apply to all splits.

In [ ]:
X_mean = X_train.mean(axis=0).astype(np.float32)
X_std  = X_train.std(axis=0).astype(np.float32) + 1e-8
Y_mean = Y_train.mean(axis=0).astype(np.float32)
Y_std  = Y_train.std(axis=0).astype(np.float32) + 1e-8

# Normalize
X_train_n = (X_train - X_mean) / X_std
X_val_n   = (X_val   - X_mean) / X_std
X_test_n  = (X_test  - X_mean) / X_std
Y_train_n = (Y_train - Y_mean) / Y_std
Y_val_n   = (Y_val   - Y_mean) / Y_std
Y_test_n  = (Y_test  - Y_mean) / Y_std

# Register stats with MPC module so inference normalizes correctly
set_normalization_stats(X_mean, X_std, Y_mean, Y_std)

# Save stats alongside weights so simulation can reload them
np.savez(NORM_STATS, X_mean=X_mean, X_std=X_std, Y_mean=Y_mean, Y_std=Y_std)
print(f"Normalization stats saved to {NORM_STATS}")

## 5. DataLoaders

In [ ]:
train_loader = DataLoader(HopperDataset(X_train_n, Y_train_n), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(HopperDataset(X_val_n,   Y_val_n),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(HopperDataset(X_test_n,  Y_test_n),  batch_size=BATCH_SIZE, shuffle=False)

## 6. Model

In [ ]:
model = HopperMLP(
    input_dim=X.shape[1],
    output_dim=Y.shape[1],
    hidden_dim=HIDDEN_DIM,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = ReduceLROnPlateau(optimizer, patience=LR_PATIENCE, factor=LR_FACTOR, verbose=True)
criterion = nn.MSELoss()

print_model_summary(model)

## 7. Training Loop

In [ ]:
train_losses, val_losses = [], []
best_val_loss = float('inf')

print("Starting training...\n")

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    epoch_train_loss = 0.0
    for X_batch, Y_batch in train_loader:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), Y_batch)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * X_batch.size(0)
    epoch_train_loss /= len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    # --- Validate ---
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            epoch_val_loss += criterion(model(X_batch), Y_batch).item() * X_batch.size(0)
    epoch_val_loss /= len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    scheduler.step(epoch_val_loss)

    # Save best
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        model.save(MODEL_WEIGHTS)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Train: {epoch_train_loss:.6f} | "
              f"Val: {epoch_val_loss:.6f} | "
              f"Best: {best_val_loss:.6f}")

print(f"\n✓ Training complete. Best val loss: {best_val_loss:.6f}")
print(f"✓ Best weights saved to {MODEL_WEIGHTS}")

## 8. Training Curves

In [ ]:
plot_training_history(train_losses, val_losses)

## 9. Test Set Evaluation

In [ ]:
# Reload best weights before evaluating
model.load(MODEL_WEIGHTS, device=device)

test_metrics = compute_metrics(model, test_loader, device=device)
print("Test Set Metrics (normalized space):")
print(f"  MSE: {test_metrics['mse']:.6f}")
print(f"  MAE: {test_metrics['mae']:.6f}")

## 10. Rollout Evaluation

Two modes are compared:
- **Teacher-forced** (original): uses ground truth state at each step — optimistic upper bound
- **Autonomous**: propagates model's own predictions — reflects real closed-loop performance

In [ ]:
HORIZON = 50
state_labels = ['pos_x', 'pos_y', 'pos_z', 'roll', 'pitch', 'yaw']

# Teacher-forced rollout (uses ground truth at each step)
rollout_tf = evaluate_rollout(model, X_test_n, Y_test_n, horizon=HORIZON, device=device)
print(f"Teacher-forced MSE ({HORIZON} steps): {rollout_tf['mse']:.6f}")
plot_rollout_predictions(rollout_tf, state_labels=state_labels)

In [ ]:
# Autonomous rollout (propagates model's own predictions)
# NOTE: this is the honest metric — expect higher error than teacher-forced
import torch

model.eval()
state = X_test_n[0, :6].copy()  # [pos(3), eul(3)]
action_context = X_test_n[0, 6:].copy()  # [thrust, tau, signals]

auto_preds, auto_gt = [], []

with torch.no_grad():
    for t in range(HORIZON):
        x_in = np.concatenate([state, X_test_n[t, 6:]])  # use real actions
        x_t  = torch.tensor(x_in, dtype=torch.float32, device=device).unsqueeze(0)
        delta = model(x_t).cpu().numpy()[0]

        state_pred = state + delta
        state_true = X_test_n[t, :6] + Y_test_n[t]

        auto_preds.append(state_pred.copy())
        auto_gt.append(state_true.copy())
        state = state_pred  # propagate model's own prediction

auto_preds = np.array(auto_preds)
auto_gt    = np.array(auto_gt)
auto_mse   = np.mean((auto_preds - auto_gt) ** 2)

print(f"Autonomous MSE ({HORIZON} steps): {auto_mse:.6f}")
print(f"(Teacher-forced was {rollout_tf['mse']:.6f} — ratio: {auto_mse/rollout_tf['mse']:.1f}x)")